# `mboostrs` — model selection

The fitted boosting model depends on the stopping iteration, learning rate, and candidate predictors:

1. **`mstop`** — how many boosting iterations to run (the key tuning parameter; too few = underfit, too many = overfit)
2. **`nu`** — the learning rate (step size shrinkage;
smaller `nu` generally requires larger `mstop`)
3. **Which predictors to include** — especially in high dimensions

This notebook shows methods for choosing these settings:

1. **Cross-validation** (bootstrap, kfold, subsampling)
2. **AIC** / corrected AIC (Hurvich-Clifford) / gMDL
3. **Stability selection** (selection frequencies across subsamples)
4. **Model comparison** (GLM vs GAM)
5. **Grid search** over `nu` and `mstop`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
import pandas as pd
import altair as alt
import mboostrs as mb

from mboostrs.datasets import bodyfat

rng = np.random.default_rng(42)
print("mboostrs version:", mb.__version__)

## Data: bodyfat

The `bodyfat` dataset contains body fat measurements and anthropometric predictors. It provides a small-p regression example for comparing model-selection methods.

In [ ]:
bf = bodyfat()
data = bf.rename({"DEXfat": "y"})
print(f"n={len(data)}, predictors: {[c for c in data.columns if c != 'y']}")

## 1. Cross-validation — three schemes

`cvrisk` is the primary tool for choosing `mstop`. It evaluates the model at every iteration along the boosting path on held-out data, returning the risk path and the optimal stopping point.

`mboostrs` supports three CV schemes:

| Scheme | Description | When to use |
|---|---|---|
| **k-fold** | Partition data into $B$ folds, hold out one at a time | Use when folds represent the intended prediction task |
| **bootstrap** | Resample with replacement $B$ times; OOB is test set | Repeated bootstrap samples with out-of-bag evaluation |
| **subsampling** | Draw $prob \cdot n$ observations without replacement | Compromise; controls training set size |

We fit a full `glmboost` with all predictors, then run all three CV schemes. You can also pass **pre-computed folds** for reproducibility — useful when comparing multiple models on the same folds.

In [ ]:
# Fit a GLM with all predictors
fit_glm = mb.glmboost(
    "y ~ age + waistcirc + hipcirc + elbowbreadth + kneebreadth + anthro3a + anthro3b + anthro3c + anthro4",
    data=data, control=mb.boost_control(mstop=500, nu=0.1),
)

# k-fold CV (the default)
cvr_kfold = fit_glm.cvrisk(type="kfold", B=10, seed=42)
print(f"kfold:        mstop*={cvr_kfold.optimal_mstop}")

# Bootstrap CV
cvr_boot = fit_glm.cvrisk(type="bootstrap", B=25, seed=42)
print(f"bootstrap:    mstop*={cvr_boot.optimal_mstop}")

# Subsampling CV (50% subsample)
cvr_sub = fit_glm.cvrisk(type="subsampling", B=25, prob=0.5, seed=42)
print(f"subsampling:  mstop*={cvr_sub.optimal_mstop}")

# Apply the optimal mstop from kfold
m_opt = cvr_kfold.optimal_mstop
fitted_opt = fit_glm[:m_opt]
print(f"Fit at mstop*={m_opt}: final risk={fitted_opt.risk_[-1]:.4f}")

# Pre-computed folds for reproducibility
fold_matrix = mb.cv_weights(len(data), type="kfold", B=5, seed=0)
print(f"Fold matrix shape: {fold_matrix.shape}")
cvr_folds = fit_glm.cvrisk(folds=fold_matrix)
print(f"With pre-computed folds: mstop*={cvr_folds.optimal_mstop}")

In [ ]:
# CV risk path comparison: kfold, bootstrap, subsampling
cv_df = pl.DataFrame({
    "iteration": list(range(len(cvr_kfold.aggregate))),
    "kfold": cvr_kfold.aggregate,
    "bootstrap": np.interp(
        np.arange(len(cvr_kfold.aggregate)),
        np.arange(len(cvr_boot.aggregate)), cvr_boot.aggregate
    ),
    "subsampling": np.interp(
        np.arange(len(cvr_kfold.aggregate)),
        np.arange(len(cvr_sub.aggregate)), cvr_sub.aggregate
    ),
}).to_pandas().melt(id_vars="iteration", var_name="scheme", value_name="risk")

alt.Chart(cv_df).mark_line().encode(
    x="iteration:Q", y="risk:Q", color="scheme:N",
    strokeDash="scheme:N",
).properties(title="CV risk path comparison \u2014 three schemes")


## 2. Gaussian AIC and gMDL

For the Gaussian squared-error model below, mboostrs supports corrected
AIC and gMDL. Both score the fitted path using a complexity penalty and
can choose different stopping iterations. Compare the selected models;
there is no guarantee that one criterion always selects fewer terms.

`method="classical"` is reserved for supported likelihood families such
as Poisson. It is rejected for `Gaussian` and `GaussReg`, whose squared-error
risk is not a negative log-likelihood. The following comparison therefore
uses only the criteria supported by this Gaussian fit.


In [ ]:
# Compare stopping criteria on one Gaussian boosting path.
fit_aic = mb.glmboost(
    "y ~ waistcirc + hipcirc + elbowbreadth + kneebreadth + anthro3a + anthro3b + anthro3c + anthro4",
    data=data, control=mb.boost_control(mstop=500, nu=0.1),
)

aic_corrected = fit_aic.aic(method="corrected")
aic_gmdl = fit_aic.aic(method="gmdl")
print(f"Corrected AIC: mstop*={aic_corrected.mstop}")
print(f"gMDL:          mstop*={aic_gmdl.mstop}")

fit_corrected = fit_aic[:aic_corrected.mstop]
fit_gmdl = fit_aic[:aic_gmdl.mstop]
print(f"Corrected: {len(fit_corrected.coefficients_)} selected predictors")
print(f"gMDL:      {len(fit_gmdl.coefficients_)} selected predictors")


In [ ]:
# The criteria have different scales; read each curve's minimum separately.
aic_df = pl.DataFrame({
    "iteration": list(range(len(aic_corrected.aic))),
    "corrected": aic_corrected.aic,
    "gmdl": aic_gmdl.aic,
}).unpivot(index="iteration", variable_name="method", value_name="criterion")

alt.Chart(aic_df).mark_line().encode(
    x="iteration:Q", y="criterion:Q", color="method:N",
    strokeDash="method:N",
).properties(title="Gaussian stopping criteria: corrected AIC and gMDL")


## 3. Stability Selection

**Stability selection** (Meinshausen & Bühlmann, 2010) measures how frequently predictors are selected across subsamples.

Under its assumptions, the selection threshold and the number of selected variables per subsample determine a bound on the expected number of false selections, the Per-Family Error Rate (PFER).

**How it works:**
1. Draw $B$ subsamples (or bootstrap samples)
2. Run boosting on each subsample
3. Count how often each variable is selected at a given `q`
4. Variables selected in $\geq$ `cutoff` fraction of runs are kept, or the PFER bound is used to determine the threshold automatically

**When to use:** high-dimensional settings ($p \gg n$) where you need to control false discoveries, or when you want interpretable results with rigorous error control.

We set up a semi-synthetic high-dimensional problem with $n=200$, $p=50$, of which only **4** are truly relevant.

In [ ]:
# Build a high-dimensional example
n = 200
p = 50
X = rng.normal(size=(n, p))
true_idx = [2, 10, 25, 40]
beta_true = np.zeros(p)
beta_true[true_idx] = [1.5, -0.8, 0.6, -0.4]
y = X @ beta_true + rng.normal(0, 0.3, n)
data_hd = pl.DataFrame(
    {f"x{j}": X[:, j] for j in range(p)} | {"y": y}
)
formula = "y ~ " + " + ".join(f"x{j}" for j in range(p))

fit_hd = mb.glmboost(
    formula, data=data_hd,
    control=mb.boost_control(mstop=200, nu=0.1),
)

print(f"Full model: {sum(1 for v in fit_hd.coefficients_.values() if abs(v) > 1e-6)} selected")

# Stability selection with PFER bound (controls expected false positives)
ss_pfer = mb.stabsel(
    fit_hd, PFER=2.0, q=10, B=50, assumption="unimodal",
    sampling_type="SS", verbose=True,
)
print(f"Stability selection (PFER): {len(ss_pfer.selected)} selected out of {ss_pfer.p}")
print(f"Selected indices: {sorted(ss_pfer.selected)}")
print(f"True active indices: {sorted(true_idx)}")

# Stability selection with cutoff threshold
ss_cutoff = mb.stabsel(
    fit_hd, cutoff=0.9, q=8, B=50, assumption="unimodal",
    sampling_type="SS", verbose=False,
)
print(f"Stability selection (cutoff=0.9): {len(ss_cutoff.selected)} selected")

# Stability selection also works for GAM with mixed terms
fit_gam_ss = mb.gamboost(
    "y ~ bbs(x, df=4) + bols(z1) + bols(z2) + bols(z3) + bols(z4) + bols(z5)",
    data={"x": rng.uniform(0, 1, n), "z1": rng.normal(size=n),
          "z2": rng.normal(size=n), "z3": rng.normal(size=n),
          "z4": rng.normal(size=n), "z5": rng.normal(size=n),
          "y": rng.normal(size=n)},
    control=mb.boost_control(mstop=100, nu=0.1),
)
ss_gam = mb.stabsel(
    fit_gam_ss, cutoff=0.9, q=3, B=30,
    assumption="unimodal", verbose=False,
)
print(f"GAM stabsel: {len(ss_gam.selected)} selected of {ss_gam.p}")

## 4. Model comparison: GLM vs GAM

Should you use linear effects (`bols`) or smooth effects (`bbs`)? The bodyfat dataset gives a clear answer: hip circumference has a non-linear relationship with body fat, so the GAM outperforms the GLM.

We compare using cross-validated risk at the optimal `mstop`.

In [ ]:
# GLM (all linear)
m_glm = mb.glmboost(
    "y ~ waistcirc + hipcirc + elbowbreadth + kneebreadth",
    data=data, control=mb.boost_control(mstop=500, nu=0.1),
)
cvr_glm = m_glm.cvrisk(type="kfold", B=10, seed=0)
m_glm_opt = m_glm[:cvr_glm.optimal_mstop]
risk_glm = m_glm_opt.risk_[-1]

# GAM (smooth terms for continuous predictors)
m_gam = mb.gamboost(
    "y ~ bbs(waistcirc, df=4) + bbs(hipcirc, df=4) + bols(elbowbreadth) + bols(kneebreadth)",
    data=data, control=mb.boost_control(mstop=500, nu=0.1),
)
cvr_gam = m_gam.cvrisk(type="kfold", B=10, seed=0)
m_gam_opt = m_gam[:cvr_gam.optimal_mstop]
risk_gam = m_gam_opt.risk_[-1]

print(f"GLM: mstop*={cvr_glm.optimal_mstop}, risk={risk_glm:.4f}")
print(f"GAM: mstop*={cvr_gam.optimal_mstop}, risk={risk_gam:.4f}")

## 5. Grid search over `nu` and `mstop`

The learning rate `nu` and stopping iteration `mstop` interact: smaller `nu` needs larger `mstop`. The best combination depends on the data.

A grid search over `nu` values (each internally cross-validated for `mstop`) lets us pick both jointly.

In [ ]:
nus = [0.05, 0.1, 0.2, 0.4]
mstop_max = 500
candidates = []

for nu in nus:
    fit_g = mb.glmboost(
        "y ~ waistcirc + hipcirc + elbowbreadth + kneebreadth",
        data=data, control=mb.boost_control(mstop=mstop_max, nu=nu),
    )
    cvr_g = fit_g.cvrisk(type="kfold", B=5, seed=0)
    candidates.append((nu, cvr_g.optimal_mstop, cvr_g.aggregate[cvr_g.optimal_mstop]))
    print(f"nu={nu:.2f}:  mstop*={cvr_g.optimal_mstop:3d}  CV risk={cvr_g.aggregate[cvr_g.optimal_mstop]:.4f}")

best_nu, best_m, best_risk = min(candidates, key=lambda x: x[2])
print(f"\nBest: nu={best_nu}, mstop*={best_m}, CV risk={best_risk:.4f}")

In [ ]:
# Grid search results: CV risk by nu
grid_df = pd.DataFrame(candidates, columns=["nu", "mstop", "cv_risk"])

base = alt.Chart(grid_df).encode(
    x=alt.X("nu:Q", title="Learning rate \u03bd"),
    y=alt.Y("mstop:Q", title="Optimal mstop"),
    size=alt.Size("cv_risk:Q", scale=alt.Scale(reverse=True)),
)
points = base.mark_point(filled=True).encode(
    color=alt.Color("cv_risk:Q", scale=alt.Scale(scheme="viridis", reverse=True)),
    tooltip=["nu", "mstop", "cv_risk"],
).properties(title="Grid search over \u03bd and mstop")
labels = base.mark_text(dy=-10).encode(text=alt.Text("cv_risk:Q", format=".1f"))
points + labels

## 6. Cross-validation diagnostics

The `CvRiskResult` object carries the full risk matrix (folds × iterations) and aggregate path. You can inspect these for diagnostics:

In [ ]:
cvr = fit_glm.cvrisk(type="kfold", B=10, seed=42)
print(f"Risk matrix shape: {cvr.risk_matrix.shape}")
print(f"Grid length: {len(cvr.grid)}")
print(f"Grid (first few): {cvr.grid[:5]}...")
print(f"Aggregate at optimal: {cvr.aggregate[cvr.optimal_mstop]:.4f}")
print(f"Optimal mstop: {cvr.optimal_mstop}")
print(f"(dtype: {cvr.risk_matrix.dtype})")

In [ ]:
# Per-fold CV risk traces.
#
# Altair refuses a chart over more than 5000 rows unless told otherwise,
# and 10 folds x 501 iterations is 5010. `mboostrs.plot` lifts that limit
# for its own charts; a notebook that builds one with bare Altair asks for
# itself. (`cvr.plot()` draws the same thing and needs none of this.)
alt.data_transformers.enable("default", max_rows=None)

fold_traces = []
for fold_idx in range(cvr.risk_matrix.shape[0]):
    fold_traces.append(pl.DataFrame({
        "iteration": list(range(cvr.risk_matrix.shape[1])),
        "risk": cvr.risk_matrix[fold_idx],
        "fold": f"fold {fold_idx}"
    }))
fold_df = pl.concat(fold_traces).to_pandas()
agg_df = pd.DataFrame({"iteration": list(range(len(cvr.aggregate))), "risk": cvr.aggregate, "fold": "aggregate"})

lines = alt.Chart(fold_df).mark_line(opacity=0.3).encode(
    x="iteration:Q", y="risk:Q", color="fold:N",
)
agg = alt.Chart(agg_df).mark_line(color="black", strokeWidth=3).encode(
    x="iteration:Q", y="risk:Q",
).properties(title="Per-fold CV risk traces (gray) + aggregate (black)")
lines + agg


### Plotting functions available

`mboostrs` provides built-in plotting for model selection:

- `fit.plot_risk()` — in-bag risk path (training error vs iteration)
- `cvr.plot()` — CV risk with individual fold traces. The plotting functions live in `mb.plot`, not at the top level, and every result object carries `.plot()`.
- `mb.plot.plot_aic(aic)` — AIC trace across iterations
- `mb.plot.plot_stabsel(ss)` — selection probabilities per variable

These are integrated with Altair for interactive visualisation.